# PAWL-Net: data preparation

This notebook uses the shared package rather than defining a second dataset or model.
Install from the repository root with `python -m pip install -e ".[notebooks,gee]"`.
Start Jupyter from the repository root or `notebooks/`.

The original Colab notebooks, including their saved outputs, are preserved locally in
`.local/original_files/`. No account identifiers or Google Drive paths are required here.
See [the data guide](../docs/data.md) and [reproducibility notes](../docs/reproducibility.md).

In [ ]:
from pawl_net.config import load_config
from pawl_net.normalization import CHANNELS

config = load_config()
ROOT = config["project_root"]
RAW_DIR = ROOT / "data" / "raw"
print("Repository:", ROOT)
print("Channel order:", CHANNELS)
print("Paths:", config["paths"])

## Optional Earth Engine export

The supplied workflow reads precomputed annual assets. Creating those assets from raw
Landsat imagery is an upstream step not included in the source material. You need access
to the original assets or equivalent assets with the documented bands.

Set `GEE_PROJECT_ID` and `GEE_ASSET_ROOT` in your shell, then run:

```text
python scripts/export_earth_engine.py --authenticate --years 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024
```

The ROI asset defaults to `<asset-root>/00_Study_ROIs` and must include `city_name`.
Training assets use `Train_L8_Year_YYYY`; held-out years use `Val_L8_Year_YYYY`.
The export function does not guess or repair scale/offset metadata in existing assets.

In [ ]:
import subprocess
import sys

# Display options without authenticating or starting an export.
subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "export_earth_engine.py"), "--help"], check=True
);

## Tile annual arrays

Annual arrays have shape `(12, H, W, 9)` and filenames `City_YYYY.npy`.
Patches use size 128 and stride 64. Years 2015–2022 go to training/validation,
and 2023–2024 go to test. Patches with less than 10% nonzero LST are rejected.
Existing patches are skipped. The internal train/validation split remains the
original sorted-file 80/20 split.

In [ ]:
from pawl_net.preprocessing import prepare_directory

annual_files = sorted(RAW_DIR.glob("*.npy"))
if annual_files:
    summary = prepare_directory(RAW_DIR, config, patch_size=128, stride=64)
    print(summary)
else:
    print("No annual arrays supplied. Place City_YYYY.npy files in", RAW_DIR)

## Inspect a local patch before normalization

This inspection displays the raw channel distribution, including source NoData.
Clipping follows the historical implementation; do not infer a new scale or offset from
an individual patch. Full archive statistics are recorded in `docs/data.md`.

In [ ]:
import numpy as np

patch_files = sorted(config["paths"]["train_patch_dir"].glob("*.npy"))
if not patch_files:
    patch_files = sorted(config["paths"]["test_patch_dir"].glob("*.npy"))

if patch_files:
    sample_path = patch_files[0]
    sample = np.load(sample_path, allow_pickle=False)
    print(sample_path.name, sample.shape, sample.dtype)
    for index, name in enumerate(CHANNELS):
        print(name, float(sample[:, :, index].min()), float(sample[:, :, index].max()))
else:
    print("No local patches. See docs/data.md for preparation instructions.")

In [ ]:
import matplotlib.pyplot as plt

from pawl_net.dataset import UHIDataset

if patch_files:
    dataset = UHIDataset(sample_path.parent, mode="test", seed=42, mask_strategy="squares")
    inputs, target, mask = dataset[0]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for axis, array, title in zip(
        axes,
        [inputs[0], target[0], mask[0]],
        ["Masked normalized LST", "Normalized target", "Visibility mask"],
    ):
        axis.imshow(array, cmap="viridis")
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()